## Fitting an unbroken power law to the HESE 7.5 year data

In [1]:
# Loading the HESE 7.5 year data

import numpy as np
import GollumFitPy as gf
import scipy.stats as stats
import os 

import sys
sys.path.append("Data/HESE/HESE 7.5 Data Release")
import data_loader 

In [ ]:
# Importing the data

HESE_75 = data_loader.load_data("Data/HESE/HESE 7.5 Data Release/resources/data/HESE_data.json")

HESE_75_Energy = np.array([i[0] for i in HESE_75])
HESE_75_Zenith = np.array([i[1] for i in HESE_75])
HESE_75_EventType = np.array([i[3] for i in HESE_75])

In [24]:
# Reformatting the data for Gollum

HESE_reformatted = np.zeros((len(HESE_75), 3))
HESE_reformatted[:, 0] = HESE_75_Energy
HESE_reformatted[:, 1] = HESE_75_Zenith
HESE_reformatted[:, 2] = HESE_75_EventType

# Choosing the events with zenith angle greater than 90 degrees (upgoing events)
# Data is in radians, so use np.pi / 2
mask = HESE_reformatted[:, 1] > (np.pi / 2)
HESE_reformatted = HESE_reformatted[mask]

# events with energy less than 1e5 GeV 

mask = HESE_reformatted[:, 0] < 2e5
HESE_reformatted = HESE_reformatted[mask]

# Saving the reformatted data

np.savez("Data/GollumFit_Data/HESE_7.5_year.npz", HESE_75=HESE_reformatted)

In [25]:
np.load("Data/GollumFit_Data/HESE_7.5_year.npz")['HESE_75']

array([[1.71397406e+05, 1.96363652e+00, 0.00000000e+00],
       [8.60804453e+04, 2.93880272e+00, 0.00000000e+00],
       [1.86558625e+05, 2.06770587e+00, 0.00000000e+00],
       [1.86010953e+05, 1.83494008e+00, 1.00000000e+00],
       [8.51670703e+04, 1.66738236e+00, 0.00000000e+00],
       [7.18423125e+04, 1.59897041e+00, 1.00000000e+00],
       [6.66631406e+04, 2.52316999e+00, 0.00000000e+00],
       [7.93988672e+04, 2.61452961e+00, 0.00000000e+00],
       [1.54762219e+05, 1.61540174e+00, 0.00000000e+00],
       [1.48418297e+05, 2.64262080e+00, 0.00000000e+00],
       [7.75298203e+04, 1.65169764e+00, 0.00000000e+00],
       [1.02387461e+05, 1.66836500e+00, 1.00000000e+00],
       [1.47270875e+05, 1.75271511e+00, 1.00000000e+00],
       [1.28198367e+05, 2.14949894e+00, 1.00000000e+00],
       [1.31973578e+05, 1.59086573e+00, 0.00000000e+00],
       [1.79897547e+05, 1.76933348e+00, 0.00000000e+00],
       [1.14974000e+05, 2.59269905e+00, 0.00000000e+00]])

---

### Initializing FastMC

In [27]:
#####################################################################################
# Configure Data Paths - Set paths for cross section splines
#####################################################################################
datapaths = gf.DataPaths()

gollum_dir = "GollumFit/GollumFit"

datapaths.neutrino_cc_xs_spline_path             = gollum_dir + "/resources/Splines/CrossSections/sigma_nu_CC_iso.fits"
datapaths.antineutrino_cc_xs_spline_path         = gollum_dir + "/resources/Splines/CrossSections/sigma_nubar_CC_iso.fits"
datapaths.neutrino_nc_xs_spline_path             = gollum_dir + "/resources/Splines/CrossSections/sigma_nu_NC_iso.fits"
datapaths.antineutrino_nc_xs_spline_path         = gollum_dir + "/resources/Splines/CrossSections/sigma_nubar_NC_iso.fits"
datapaths.diff_neutrino_cc_xs_spline_path        = gollum_dir + "/resources/Splines/CrossSections/dsdxdy_nu_CC_iso.fits"
datapaths.diff_antineutrino_cc_xs_spline_path    = gollum_dir + "/resources/Splines/CrossSections/dsdxdy_nubar_CC_iso.fits"
datapaths.diff_neutrino_nc_xs_spline_path        = gollum_dir + "/resources/Splines/CrossSections/dsdxdy_nu_NC_iso.fits"
datapaths.diff_antineutrino_nc_xs_spline_path    = gollum_dir + "/resources/Splines/CrossSections/dsdxdy_nubar_CC_iso.fits"
datapaths.mc_path                                = gollum_dir + "/monte_carlo/"
datapaths.domeff_spline_path                     = gollum_dir + "/resources/Splines/DOMEffSplines/new_ddmnodeis/BDT/DnnEnergy_0.99"
datapaths.holeice_spline_path                    = gollum_dir + "/resources/Splines/HoleIceSplines/new_ddmnodeis/BDT/DnnEnergy_0.99"
datapaths.attenuation_spline_path                = gollum_dir + "/resources/Splines/AttenuationSplines/new_ddmnodeis"
datapaths.ice_gradient_spline_path               = gollum_dir + "/resources/Splines/IceGradientsSplines/new_ddmnodeis/BDT/DnnEnergy_0.99"
datapaths.atmospheric_density_spline_path        = gollum_dir + "/resources/Splines/AtmosphericZenithVariationSplines/atm_density_1s.fits"
datapaths.atmospheric_kaonlosses_spline_path     = gollum_dir + "/resources/Splines/AtmosphericKaonLossesSplines/kaon_loses_1s.fits"


In [28]:
#####################################################################################
# Configure Flux Files - Load atmospheric, prompt, and astrophysical flux files
#####################################################################################
datapaths.conventional_nusquids_atmospheric_file = gollum_dir + "/examples/fluxes/atmospheric.hdf5"
datapaths.prompt_nusquids_atmospheric_file       = gollum_dir + "/examples/fluxes/prompt_atmospheric.hdf5"
datapaths.astro_nusquids_file                    = gollum_dir + "/examples/fluxes/astro.hdf5"

# Hadronic and cosmic ray correction splines (necessary for flux nuisance parameters)
hadronlist = ["he_K+", "he_K-", "vhe1_pi+", "vhe1_pi-", "vhe3_K+", "vhe3_K-", 
              "vhe3_pi+", "vhe3_pi-", "vhe3_p", "vhe3_n"]
crlist = ["GSF_1", "GSF_2", "GSF_3", "GSF_4", "GSF_5", "GSF_6"]

datapaths.hadronic_spline_path   = gollum_dir + "/examples/fluxes"
datapaths.cosmic_ray_spline_path = gollum_dir + "/examples/fluxes"

In [29]:
#####################################################################################
# Set Steering Parameters - Configure analysis binning and settings
# NOTE: Binning choices affect FastMC compression and must match analysis configuration
#####################################################################################
steering_params = gf.SteeringParams()
steering_params.minFitEnergy                    = 60000
steering_params.maxFitEnergy                    = 2e5
steering_params.logEbinEdge                     = np.log10(60000)
steering_params.logEbinWidth                    = (np.log10(2e5) - np.log10(60000)) / 24
steering_params.minCosth                        = -1.0
steering_params.maxCosth                        = 0.0
steering_params.cosThbinEdge                    = 0.0
steering_params.cosThbinWidth                   = 0.05
steering_params.selectionStart                  = 0.99
steering_params.ice_gradient_filename           = ["Amp_0", "Amp_1", "Amp_2", "Amp_3", "Amp_4", 
                                                   "Phs_1", "Phs_2", "Phs_3", "Phs_4"]
steering_params.active_hadronic_parameters      = hadronlist
steering_params.active_cosmicray_parameters     = crlist

# Livetime for the corresponding Monte Carlo
years = 7.5
steering_params.fullLivetime                    = years * 365 * 24 * 60 * 60.
steering_params.simToLoad                       = "BDT_Split_HE"
steering_params.energyName                      = "DnnEnergy"
steering_params.model_label                     = ""  # Can be used for uniquely-labelled flux files

In [30]:
#####################################################################################
# Construct and Write FastMC
#####################################################################################
gollumfit = gf.GollumFit(datapaths, steering_params)

# Compression parameter: smaller values = higher compression but potential accuracy loss
metascaling = 0.25
gollumfit.ConstructFastMode(metascaling)

# Write to file
gollumfit.WriteCompact("Data/GollumFit_Data/compact.fastmc")

print("Done generating FastMC.")

reset_steering: 1
reset_data: 1
Loading DOM efficiency splines...
Checking file from path GollumFit/GollumFit/resources/Splines/DOMEffSplines/new_ddmnodeis/BDT/DnnEnergy_0.99/domefficiency_spline_stacked_atmConv_track.fits
Checking file from path GollumFit/GollumFit/resources/Splines/DOMEffSplines/new_ddmnodeis/BDT/DnnEnergy_0.99/domefficiency_spline_stacked_atmConv_shower.fits
Checking file from path GollumFit/GollumFit/resources/Splines/DOMEffSplines/new_ddmnodeis/BDT/DnnEnergy_0.99/domefficiency_spline_stacked_atmPrompt_track.fits
Checking file from path GollumFit/GollumFit/resources/Splines/DOMEffSplines/new_ddmnodeis/BDT/DnnEnergy_0.99/domefficiency_spline_stacked_atmPrompt_shower.fits
Checking file from path GollumFit/GollumFit/resources/Splines/DOMEffSplines/new_ddmnodeis/BDT/DnnEnergy_0.99/domefficiency_spline_stacked_diffuseAstro_track.fits
Checking file from path GollumFit/GollumFit/resources/Splines/DOMEffSplines/new_ddmnodeis/BDT/DnnEnergy_0.99/domefficiency_spline_stacked_

---

### Generating and minimizing likelihood

In [31]:
#####################################################################################
# Define Nuisance Parameters (All set to vary in fit with False flag)
# Format: [vary_flag, prior_type, center, width, lower_bound, upper_bound]
#####################################################################################

syst_dict     = { 
    'convNorm'                  : [ True, 'Gaussian',      1.,   0.2,                   0.1,                   3. ], 
    'zenithCorrection'          : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'kaonLosses'                : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'hadronicHEkp'              : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicHEkm'              : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE1pip'           : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE1pim'           : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE3kp'            : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE3km'            : [ True, 'Gaussian',      0.,    1.,                  -1.5,                   2. ], 
    'hadronicVHE3pip'           : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE3pim'           : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE3p'             : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE3n'             : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'cosmicRay1'                : [ True, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'cosmicRay2'                : [ True, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'cosmicRay3'                : [ True, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'cosmicRay4'                : [ True, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'cosmicRay5'                : [ True, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'cosmicRay6'                : [ True, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'icegrad0'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad1'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad2'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad3'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad4'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad5'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad6'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad7'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad8'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'domEfficiency'             : [ True, 'Gaussian',    1.27, 0.123,                 1.234,                1.346 ], 
    'holeiceForward'            : [ True, 'Gaussian',     -1.,   10.,                 -5.35,                 1.85 ], 
    'astroNorm'                 : [ False, 'Gaussian', 4.72/6.,  0.36,                    0.,                   3. ], 
    'astroDeltaGamma'           : [ False, 'Gaussian',      0.,  0.36,                   -2.,                   2. ], 
    'astroDeltaGammaSec'        : [ True, 'Gaussian',      0.,  0.36,                   -2.,                   2. ], 
    'nuxs'                      : [ True, 'Gaussian',      1.,   0.1,                 0.824,                1.176 ], 
    'nubarxs'                   : [ True, 'Gaussian',      1.,   0.1,                 0.824,                1.176 ], 
    'astroPivot'                : [ False,  'Uniform',      5.,    1.,                    4.,                   6. ], 
    'promptNorm'                : [ True, 'Gaussian',      1.,    1.,                    0.,                   3. ],
    'NeutrinoAntineutrinoRatio' : [ True, 'Gaussian',      1.,    1.,                    0.,                   2. ],
}

In [33]:
#####################################################################################
# Define Random Sampling Function
# Helper function to sample random parameter values from prior distributions
#####################################################################################
def throw(syst):
    """Sample random value from prior distribution."""
    if syst[1] == 'Gaussian':
        # Truncated normal distribution
        val = stats.truncnorm(
            (syst[4] - syst[2]) / syst[3],  # Lower bound (standardized)
            (syst[5] - syst[2]) / syst[3],  # Upper bound (standardized)
            syst[2],  # Mean
            syst[3]   # Std dev
        ).rvs(1)
        return val[0]
    else:
        # Uniform distribution
        return np.random.uniform(syst[4], syst[5])

In [34]:
#####################################################################################
# Initialize Fit Parameter Objects
# Create objects to manage fit configuration
#####################################################################################

fitparams_flag  = gf.FitParametersFlag()  # Which parameters to vary
fitparams_bound = gf.FitParametersBound()  # Parameter bounds
priors          = gf.Priors()              # Prior distributions
seed_fitparams  = gf.FitParameters()       # Initial values

In [35]:
#####################################################################################
# Set Priors and Random Initial Values
# Configure priors and randomly initialize starting parameter values
#####################################################################################
np.random.seed(100)  # For reproducibility
print('Initializing with the following randomly-seeded nuisance params:')

for sname in syst_dict.keys():
    # Set flags and bounds
    exec(f'fitparams_flag.{sname} = syst_dict["{sname}"][0]')
    exec(f'fitparams_bound.{sname}Min = syst_dict["{sname}"][4]')
    exec(f'fitparams_bound.{sname}Max = syst_dict["{sname}"][5]')
    
    # Set priors
    if syst_dict[sname][1] == 'Gaussian':
        exec(f'priors.{sname}Center = syst_dict["{sname}"][2]')
        exec(f'priors.{sname}Width  = syst_dict["{sname}"][3]')
    else:
        exec(f'priors.{sname}Min = syst_dict["{sname}"][4]')
        exec(f'priors.{sname}Max = syst_dict["{sname}"][5]')
    
    # Randomly initialize
    thrown_val = throw(syst_dict[sname])
    exec(f'seed_fitparams.{sname} = thrown_val')
    print(f'{sname}: {thrown_val}')

Initializing with the following randomly-seeded nuisance params:
convNorm: 1.0218039025365804
zenithCorrection: -0.5859107615392637
kaonLosses: -0.18982947806267175
hadronicHEkp: 0.9505703117875254
hadronicHEkm: -1.9227730515352341
hadronicVHE1pip: -1.0857815492239715
hadronicVHE1pim: 0.4206097302152767
hadronicVHE3kp: 0.8816848379383713
hadronicVHE3km: -0.873223209272313
hadronicVHE3pip: 0.18064418246274516
hadronicVHE3pim: 1.1431734199069066
hadronicVHE3p: -0.7640003993723422
hadronicVHE3n: -0.8428868347267422
cosmicRay1: -1.2350728474922446
cosmicRay2: -0.7731552686226462
cosmicRay3: 2.025529085014589
cosmicRay4: 0.8840426547279103
cosmicRay5: -0.946441227773658
cosmicRay6: 0.9009957799235485
icegrad0: -0.5987083265003421
icegrad1: -0.17156803788272793
icegrad2: 1.5451237127171549
icegrad3: 0.9032062912304294
icegrad4: -0.4218851785443732
icegrad5: -0.9296090139817285
icegrad6: -0.32345491345719346
icegrad7: -2.4560823868999235
icegrad8: -0.664782677751945
domEfficiency: 1.321005701

In [36]:
#####################################################################################
# Set Correlations and Paths
# Load correlation matrices for ice gradients and flux parameters
#####################################################################################
gollumdir = "GollumFit/GollumFit"

# Set correlations (required for fitting/minimization, not for likelihood evaluation)
iceg_corr = np.load(gollumdir + '/resources/correlation_matrices/icegrad_correlations.npy')
flux_corr = np.load(gollumdir + '/resources/correlation_matrices/flux_correlations_new_ddmnodeis.npy')
for idx, val in np.ndenumerate(iceg_corr):
    priors.SetIceGradientsCorr(idx[0], idx[1], val)
for idx, val in np.ndenumerate(flux_corr):
    priors.SetFluxCorr(idx[0], idx[1], val)

datapaths = gf.DataPaths()
datapaths.domeff_spline_path      = gollumdir + '/resources/Splines/DOMEffSplines/new_ddmnodeis/BDT/DnnEnergy_0.99'
datapaths.holeice_spline_path     = gollumdir + '/resources/Splines/HoleIceSplines/new_ddmnodeis/BDT/DnnEnergy_0.99'
datapaths.attenuation_spline_path = gollumdir + '/resources/Splines/AttenuationSplines/new_ddmnodeis'
datapaths.compact_file_path       = 'Data/GollumFit_Data/compact.fastmc'

In [37]:
#####################################################################################
# Configure Steering Parameters
# Set binning and convergence criteria (must match FastMC binning)
#####################################################################################
edges = np.logspace(np.log10(60000), np.log10(2e5), 25)
steering_params                = gf.SteeringParams()
steering_params.minFitEnergy   = edges[0]
steering_params.maxFitEnergy   = edges[-1]
steering_params.logEbinEdge    = np.log10(edges[0])
steering_params.logEbinWidth   = np.log10(edges[1]) - np.log10(edges[0])
steering_params.minCosth       = -1.0
steering_params.maxCosth       = 0.0
steering_params.cosThbinEdge   = 0.0
steering_params.cosThbinWidth  = 0.05
steering_params.selectionStart = float("DnnEnergy_0.99".split("_")[1])
steering_params.evalThreads    = 1

# Convergence criteria (tight tolerances for accurate minimization)
steering_params.change_tol     = 1.e-20
steering_params.grad_tol       = 1.e-20
steering_params.uncertaintyModSigmaOverMu = 0.0

In [38]:
#####################################################################################
# Load Data and Configure Fit
# Create GollumFit object and load pseudo-data
#####################################################################################
gollumfit = gf.GollumFit(datapaths, steering_params)

#####################################################################################
# declare the fake data location and load it
#####################################################################################
realization  = "Data/GollumFit_Data/HESE_7.5_year.npz"
total_data = gollumfit.SetData(np.load(realization)["HESE_75"])

reset_steering: 1
reset_data: 1
Loading DOM efficiency splines...
Checking file from path GollumFit/GollumFit/resources/Splines/DOMEffSplines/new_ddmnodeis/BDT/DnnEnergy_0.99/domefficiency_spline_stacked_atmConv_track.fits
Checking file from path GollumFit/GollumFit/resources/Splines/DOMEffSplines/new_ddmnodeis/BDT/DnnEnergy_0.99/domefficiency_spline_stacked_atmConv_shower.fits
Checking file from path GollumFit/GollumFit/resources/Splines/DOMEffSplines/new_ddmnodeis/BDT/DnnEnergy_0.99/domefficiency_spline_stacked_atmPrompt_track.fits
Checking file from path GollumFit/GollumFit/resources/Splines/DOMEffSplines/new_ddmnodeis/BDT/DnnEnergy_0.99/domefficiency_spline_stacked_atmPrompt_shower.fits
Checking file from path GollumFit/GollumFit/resources/Splines/DOMEffSplines/new_ddmnodeis/BDT/DnnEnergy_0.99/domefficiency_spline_stacked_diffuseAstro_track.fits
Checking file from path GollumFit/GollumFit/resources/Splines/DOMEffSplines/new_ddmnodeis/BDT/DnnEnergy_0.99/domefficiency_spline_stacked_

In [39]:
#####################################################################################
# feed the flags, bounds, priors, on the nuisance parameters into gollumfit
#####################################################################################
gollumfit.SetFitParametersFlag(fitparams_flag)
gollumfit.SetFitParametersBound(fitparams_bound)
gollumfit.SetFitParametersPriors(priors)
gollumfit.SetFitParametersSeed([seed_fitparams])
gollumfit.ConstructLikelihoodProblem()

In [40]:
#####################################################################################
# perform the minimization
#####################################################################################
print("Starting minimization...")
min_llh = gollumfit.MinLLH()

Starting minimization...
-1.06029e+07 [1.89321e+06,61893.7,22678,38610.1,4173.62,1515.91,2251.48,2001.63,200370,106965,39813.6,16945.6,26692.6,17523.9,71279.2,30968,-18142.9,13965.2,-79392.9,29381.8,38358.7,5402.89,-5225.68,5734.33,-10422.3,89092.3,-4380.19,-25188.5,-20590.9,2.75658e+06,-8089.33,530551,909567,3.47987,102120,-140393,-537117,-39514.8]
-2.4481e+06 [1.17131e+06,1168.72,235.94,321.266,-40.9626,53.7547,-302.351,424.165,899.129,3191.96,362.504,399.783,323.09,272.06,887.913,399.028,-225.029,162.625,-967.674,318.283,17835,1319.62,-3672.41,3379.23,-5498.32,46096.1,-3284.14,-13194.5,-11814.8,988191,-2518.3,394250,3.19616e+06,3.47987,5.447e+06,1.17234e+06,-270941,-65943.3]
LH: 2.4481e+06


In [41]:
#####################################################################################
# results: print the best fit nuisance parameters, likelihood, and the number of LLH evaluations
#####################################################################################
systematics = ""
for sname in syst_dict.keys() :
    exec('print(\"'+sname+'\",min_llh.params.'+sname+')')
    exec('systematics += str(min_llh.params.'+sname+')+\" \"')

print('llh:',min_llh.likelihood)
print('nEval: '+str(min_llh.nEval))

print("Completed successfully. Bye!")

convNorm 1.021803855895996
zenithCorrection -0.5859107375144958
kaonLosses -0.189829483628273
hadronicHEkp 0.950570285320282
hadronicHEkm -1.922773003578186
hadronicVHE1pip -1.0857815742492676
hadronicVHE1pim 0.4206097424030304
hadronicVHE3kp 0.8816848397254944
hadronicVHE3km -0.8732231855392456
hadronicVHE3pip 0.1806441843509674
hadronicVHE3pim 1.1431734561920166
hadronicVHE3p -0.764000415802002
hadronicVHE3n -0.8428868055343628
cosmicRay1 -1.2350728511810303
cosmicRay2 -0.7731552720069885
cosmicRay3 2.025529146194458
cosmicRay4 0.8840426802635193
cosmicRay5 -0.9464412331581116
cosmicRay6 0.9009957909584045
icegrad0 -0.5987083315849304
icegrad1 -0.17156803607940674
icegrad2 1.5451236963272095
icegrad3 0.9032062888145447
icegrad4 -0.4218851923942566
icegrad5 -0.9296090006828308
icegrad6 -0.32345491647720337
icegrad7 -2.456082344055176
icegrad8 -0.664782702922821
domEfficiency 1.3210057020187378
holeiceForward -5.232439041137695
astroNorm 3.0
astroDeltaGamma 2.0
astroDeltaGammaSec -0.45